# 1. Реализация **LogMelFilterBanks**

In [2]:
from typing import Optional

import torch
from torch import nn
from torchaudio import functional as F

In [3]:
class LogMelFilterBanks(nn.Module):
    def __init__(
            self,
            n_fft: int = 400,
            samplerate: int = 16000,
            hop_length: int = 160,
            n_mels: int = 80,
            pad_mode: str = 'reflect',
            power: float = 2.0,
            normalize_stft: bool = False,
            onesided: bool = True,
            center: bool = True,
            return_complex: bool = True,
            f_min_hz: float = 0.0,
            f_max_hz: Optional[float] = None,
            norm_mel: Optional[str] = None,
            mel_scale: str = 'htk'
        ):
        super(LogMelFilterBanks, self).__init__()
        # general params and params defined by the exercise
        self.n_fft = n_fft
        self.samplerate = samplerate
        self.window_length = n_fft
        self.window = torch.hann_window(self.window_length)
        
        # ---- STFT params ----
        self.hop_length = hop_length
        self.n_mels = n_mels
        self.center = center
        self.return_complex = return_complex
        self.onesided = onesided
        self.normalize_stft = normalize_stft
        self.pad_mode = pad_mode
        self.power = power

        # ---- Mel filterbank params ----
        self.f_min_hz = f_min_hz
        # torchaudio convention: if f_max is None, use sample_rate // 2
        self.f_max_hz = f_max_hz if f_max_hz is not None else float(samplerate // 2)
        self.norm_mel = norm_mel
        self.mel_scale = mel_scale

        # finish parameters initialization
        self.mel_fbanks = self._init_melscale_fbanks()

    def _init_melscale_fbanks(self):
        # To access attributes, use self.<parameter_name>
        n_freqs = self.n_fft // 2 + 1
        return F.melscale_fbanks(
            n_freqs=n_freqs,
            f_min=self.f_min_hz,
            f_max=self.f_max_hz,
            n_mels=self.n_mels,
            sample_rate=self.samplerate,
            norm=self.norm_mel,
            mel_scale=self.mel_scale,
        )

    def spectrogram(self, x):
        # x - is an input signal
        return torch.stft(
            x,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.window_length,
            window=self.window,
            center=self.center,
            pad_mode=self.pad_mode,
            normalized=self.normalize_stft,
            onesided=self.onesided,
            return_complex=self.return_complex,
        )

    def forward(self, x):
        """
        Args:
            x (Torch.Tensor): Tensor of audio of dimension (batch, time), audiosignal
        Returns:
            Torch.Tensor: Tensor of log mel filterbanks of dimension (batch, n_mels, n_frames),
                where n_frames is a function of the window_length, hop_length and length of audio
        """
        # make comp stft
        stft_complex = self.spectrogram(x)

        # create power spec-m
        spec = stft_complex.abs().pow(self.power)

        # apply mel-filterbanks
        mel_spec = torch.matmul(
            spec.transpose(-1, -2), self.mel_fbanks
        ).transpose(-1, -2)

        return torch.log(mel_spec + 1e-6)

# 2. Сравнение с **torchaudio.transforms.MelSpectrogram**

In [4]:
import torchaudio
import matplotlib.pyplot as plt

In [17]:
import os
os.environ['TORCHAUDIO_USE_TORCHCODEC'] = '0'

wav_path = "test.wav" # path to test audio
signal, sr = torchaudio.load(wav_path)
assert sr == 16000, f"Expected 16 kHz, got {sr}"

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call 

Process by **torchaudio.transforms.MelSpectrogram**

In [ ]:
melspec = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=400,
    hop_length=160,
    n_mels=80,
)(signal)

log_melspec_ref = torch.log(melspec + 1e-6)

Process by **LogMelFilterBanks**

In [ ]:
logmelbanks = LogMelFilterBanks()(signal)

Check correctiveness of **LogMelFilterBanks** realisation

In [ ]:
assert log_melspec_ref.shape == logmelbanks.shape, \
    f"Shape mismatch:\n correct: {log_melspec_ref.shape}\n got: {logmelbanks.shape}"
assert torch.allclose(log_melspec_ref, logmelbanks), \
    "Values do not match!"
print("Realisation is correct")

Build visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

im0 = axes[0].imshow(
    log_melspec_ref[0].detach().numpy(),
    aspect='auto', origin='lower', interpolation='none'
)
axes[0].set_title("torchaudio  log(MelSpectrogram + ε)")
axes[0].set_xlabel("Frame")
axes[0].set_ylabel("Mel bin")
fig.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(
    logmelbanks[0].detach().numpy(),
    aspect='auto', origin='lower', interpolation='none'
)
axes[1].set_title("LogMelFilterBanks (our implementation)")
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Mel bin")
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.savefig("logmelbanks_comparison.png", dpi=150)
plt.show()